In [ ]:

# ============================================================
# HANDWRITTEN DIGIT & CAPITAL LETTER RECOGNITION
# MNIST + EMNIST + CNN + GRADIO
# ============================================================

!pip install -q tensorflow tensorflow-datasets gradio pillow

import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds
import gradio as gr

from PIL import Image, ImageOps
from tensorflow.keras import layers, models

# ============================================================
# 1. LOAD MNIST DATASET
# ============================================================

(x_tr_d, y_tr_d), (x_te_d, y_te_d) = tf.keras.datasets.mnist.load_data()

# ============================================================
# 2. LOAD EMNIST LETTERS DATASET
# ============================================================

def load_emnist_letters(split):

    ds = tfds.load(
        "emnist/letters",
        split=split,
        batch_size=-1,
        as_supervised=True
    )

    imgs, labels = tfds.as_numpy(ds)

    imgs = imgs.squeeze(-1)

    # Correct EMNIST orientation
    imgs = np.transpose(imgs, (0, 2, 1))

    labels = labels.astype(np.int64)

    # EMNIST labels start from 1
    keep = labels >= 1

    imgs = imgs[keep]
    labels = labels[keep] - 1 + 10

    return imgs, labels


x_tr_l, y_tr_l = load_emnist_letters("train")
x_te_l, y_te_l = load_emnist_letters("test")

# ============================================================
# 3. COMBINE MNIST AND EMNIST
# ============================================================

x_train = np.concatenate([x_tr_d, x_tr_l]).astype("float32") / 255.0
y_train = np.concatenate([y_tr_d, y_tr_l])

x_test = np.concatenate([x_te_d, x_te_l]).astype("float32") / 255.0
y_test = np.concatenate([y_te_d, y_te_l])

# Add channel dimension
x_train = x_train[..., np.newaxis]
x_test = x_test[..., np.newaxis]

# Shuffle training data
idx = np.random.permutation(len(x_train))

x_train = x_train[idx]
y_train = y_train[idx]

print("Training shape:", x_train.shape)
print("Testing shape:", x_test.shape)
print("Number of classes:", len(np.unique(y_train)))

# ============================================================
# 4. BUILD CNN MODEL
# ============================================================

model = models.Sequential([

    layers.Input(shape=(28, 28, 1)),

    layers.Conv2D(32, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),

    layers.Dense(128, activation="relu"),

    layers.Dropout(0.3),

    layers.Dense(36, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

# ============================================================
# 5. TRAIN MODEL
# ============================================================

history = model.fit(
    x_train,
    y_train,
    epochs=5,
    batch_size=128,
    validation_split=0.1,
    verbose=1
)

# ============================================================
# 6. EVALUATE MODEL
# ============================================================

test_loss, test_acc = model.evaluate(
    x_test,
    y_test,
    verbose=0
)

print("\nTest Accuracy:", round(test_acc * 100, 2), "%")

# Save model
model.save("handwriting_model.keras")

print("Model saved successfully!")

# ============================================================
# 7. CLASS INDEX TO CHARACTER
# ============================================================

def class_to_char(c):

    if c < 10:
        return str(c)

    return chr(ord("A") + c - 10)


# ============================================================
# 8. IMAGE PREPROCESSING
# ============================================================

def to_pil(value):

    if value is None:
        return None

    if isinstance(value, dict):
        value = value.get("composite")

        if value is None:
            return None

    if isinstance(value, np.ndarray):
        img = Image.fromarray(value)
    else:
        img = value

    if img.mode in ("RGBA", "LA"):

        bg = Image.new(
            "RGBA",
            img.size,
            (255, 255, 255, 255)
        )

        img = Image.alpha_composite(
            bg,
            img.convert("RGBA")
        )

    return img.convert("L")


# ============================================================
# 9. PREDICTION FUNCTION
# ============================================================

def predict(value):

    img = to_pil(value)

    if img is None:
        return None, {}, "Please draw or upload an image."

    # Convert black ink on white background
    img = ImageOps.invert(img)

    # Detect character boundaries
    bbox = img.getbbox()

    if bbox is None:
        return None, {}, "The image is blank. Please draw something."

    img = img.crop(bbox)

    # Resize while preserving aspect ratio
    img.thumbnail((20, 20))

    # Create 28x28 black background
    canvas = Image.new("L", (28, 28), 0)

    # Center character
    canvas.paste(
        img,
        (
            (28 - img.width) // 2,
            (28 - img.height) // 2
        )
    )

    # Normalize image
    arr = np.array(
        canvas,
        dtype=np.float32
    ).reshape(1, 28, 28, 1) / 255.0

    # Predict
    probs = model.predict(arr, verbose=0)[0]

    # Top 5 predictions
    top5 = np.argsort(probs)[::-1][:5]

    scores = {
        class_to_char(int(c)): float(probs[c])
        for c in top5
    }

    # Best prediction
    best = int(top5[0])

    kind = "Digit" if best < 10 else "Alphabet"

    # Separate digit and alphabet probabilities
    digit_prob = probs[:10].sum() * 100
    alphabet_prob = probs[10:].sum() * 100

    best_digit = int(np.argmax(probs[:10]))
    best_letter = chr(
        ord("A") + int(np.argmax(probs[10:]))
    )

    # Result summary
    result = f"""
### Prediction: {class_to_char(best)}

**Type:** {kind}

**Confidence:** {probs[best] * 100:.2f}%

### Digit vs Alphabet

- Digit probability: {digit_prob:.2f}%
- Alphabet probability: {alphabet_prob:.2f}%

### Best Predictions

- Best digit: {best_digit} ({probs[best_digit] * 100:.2f}%)
- Best alphabet: {best_letter} ({probs[10:].max() * 100:.2f}%)
"""

    return (
        canvas.resize((196, 196), Image.Resampling.NEAREST),
        scores,
        result
    )


# ============================================================
# 10. GRADIO INTERFACE
# ============================================================

with gr.Blocks(
    title="Handwritten Digit & Alphabet Recognition",
    theme=gr.themes.Soft()
) as demo:

    gr.Markdown("""
    # ✍️ Handwritten Digit & Alphabet Recognition

    Draw one digit (0-9) or one capital letter (A-Z),
    or upload an image to recognize it.

    **AI Model:** Convolutional Neural Network (CNN)
    """)

    with gr.Row():

        with gr.Column():

            with gr.Tab("Draw"):

                sketch = gr.Sketchpad(
                    type="pil",
                    image_mode="RGBA",
                    canvas_size=(320, 320),
                    label="Draw a digit or capital letter"
                )

                b1 = gr.Button(
                    "Recognize",
                    variant="primary"
                )

            with gr.Tab("Upload"):

                upload = gr.Image(
                    type="pil",
                    label="Upload handwritten image"
                )

                b2 = gr.Button(
                    "Recognize",
                    variant="primary"
                )

        with gr.Column():

            summary = gr.Markdown(
                "### Prediction results will appear here."
            )

            top5_label = gr.Label(
                num_top_classes=5,
                label="Top 5 Predictions"
            )

            seen = gr.Image(
                label="Processed Image (28x28)",
                interactive=False
            )

    outputs = [seen, top5_label, summary]

    b1.click(
        fn=predict,
        inputs=sketch,
        outputs=outputs
    )

    b2.click(
        fn=predict,
        inputs=upload,
        outputs=outputs
    )


# ============================================================
# 11. LAUNCH APPLICATION
# ============================================================

demo.launch(
    share=True,
    debug=True
)


Training shape: (148800, 28, 28, 1)
Testing shape: (24800, 28, 28, 1)
Number of classes: 36


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 13, 13, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 11, 11, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 5, 5, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       204,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 36)             │         4,644 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 228,388 (892.14 KB)

 Trainable params: 228,388 (892.14 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
1047/1047 ━━━━━━━━━━━━━━━━━━━━ 114s 107ms/step - accuracy: 0.7965 - loss: 0.6834 - val_accuracy: 0.9114 - val_loss: 0.2767
Epoch 2/5
1047/1047 ━━━━━━━━━━━━━━━━━━━━ 110s 105ms/step - accuracy: 0.9055 - loss: 0.3024 - val_accuracy: 0.9373 - val_loss: 0.2005
Epoch 3/5
1047/1047 ━━━━━━━━━━━━━━━━━━━━ 111s 106ms/step - accuracy: 0.9232 - loss: 0.2407 - val_accuracy: 0.9429 - val_loss: 0.1817
Epoch 4/5
1047/1047 ━━━━━━━━━━━━━━━━━━━━ 139s 103ms/step - accuracy: 0.9335 - loss: 0.2070 - val_accuracy: 0.9472 - val_loss: 0.1641
Epoch 5/5
1047/1047 ━━━━━━━━━━━━━━━━━━━━ 141s 103ms/step - accuracy: 0.9394 - loss: 0.1859 - val_accuracy: 0.9495 - val_loss: 0.1575

Test Accuracy: 94.23 %
Model saved successfully!


/tmp/ipykernel_1281/4057949258.py:287: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://f6b47dcdf52f841db2.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
